# Notebook 4 · Team (Modular) Multi-Agent System

> **Series:** LangChain MAS Architectures · Travel Agency Toy Use Case

---

## What is a Team Architecture?

In a **team MAS** agents have **fixed, non-overlapping roles**.
Each teammate owns exactly one section of a shared project board and
is responsible for that section from start to finish.
The workflow is a relay: one teammate completes their section,
then the next teammate picks up the enriched board and adds theirs.

```
USER REQUEST
     │
     ▼
┌──────────────┐     ┌──────────────┐     ┌──────────────┐
│  Destination │────►│    Flight    │────►│    Hotel     │
│   Designer   │     │  Specialist  │     │  Specialist  │
│ (dest+period)│     │  (flight)    │     │  (hotel)     │
└──────────────┘     └──────────────┘     └──────────────┘
                                                 │
                                                 ▼
                                     ┌──────────────────────┐
                                     │  Activity Specialist │
                                     │    (activities)      │
                                     └──────────┬───────────┘
                                                │
                                                ▼
                                     ┌──────────────────────┐
                                     │  Itinerary Writer    │
                                     │  (final write-up)    │
                                     └──────────────────────┘
          each role owns one board section · relay pattern
```

### Key Properties
| Property | Team |
|---|---|
| Authority | Distributed — each role owns its section |
| Communication | Relay through shared project board |
| Stop condition | All sections are filled (fixed pipeline) |
| Failure mode | Upstream mistakes propagate without correction |

### When to Use It
Team designs suit tasks that decompose cleanly into **non-overlapping subtasks**,
where you want each piece to be handled by the most specialised agent,
and where a clear fixed pipeline is more reliable than dynamic routing.

---

## What We Will Build

Five **teammates** each own one section of a shared project board:

| Teammate | Board section |
|---|---|
| Destination designer | `destination_and_period` |
| Flight specialist | `flight` |
| Hotel specialist | `hotel` |
| Activity specialist | `activities` |
| Itinerary writer | `final_writeup` |

Each teammate reads the board (with all earlier sections already filled),
writes their own section, and passes the updated board to the next teammate.


## 1 · Setup: One Model, One Request, One Catalog

The same three blocks appear in every notebook of this series:

| Block | Purpose |
|---|---|
| `model` | One small, cheap LLM used by every agent |
| `USER_REQUEST` | The single traveler request we want to fulfill |
| `CATALOG` + helpers | The only data source agents are allowed to use |

Keeping these identical lets you compare orchestration patterns in isolation.
Nothing changes between notebooks except *how agents talk to each other*.

> **Prerequisite:** set `OPENAI_API_KEY` in your environment before running.


In [1]:
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model
from langchain_core.messages import HumanMessage, SystemMessage

# ── One model powers every agent in the series ──────────────────────────────
# We use gpt-4.1-mini for speed and low cost during learning.
# Swap to any model that supports structured output (temperature=0 keeps
# outputs deterministic so you can rerun and compare results).
model = init_chat_model("openai:gpt-4.1-mini", temperature=0)

# ── The traveler request ─────────────────────────────────────────────────────
# All four notebooks solve this exact request.
# The only thing that changes is the orchestration architecture.
USER_REQUEST = """\
Plan a 4-day spring trip from Rome.
Requirements:
- mid-range budget
- easy flights
- central hotel
- mix of food and culture
- simple daily plan""".strip()

# ── Static catalog ────────────────────────────────────────────────────────────
# Agents must only use destinations, flights, hotels, and activities
# listed below. This constraint keeps the toy example clean and comparable.

DESTINATIONS = {
    "Lisbon":    {"best_period": "April–June", "style": "sunny, walkable, relaxed",
                  "notes": "great for food, viewpoints, and compact neighborhoods"},
    "Barcelona": {"best_period": "April–June", "style": "lively, artistic, seaside",
                  "notes": "strong mix of architecture, beach walks, and tapas"},
    "Prague":    {"best_period": "April–June", "style": "historic, compact, lower-cost",
                  "notes": "easy sightseeing with a classic old-town atmosphere"},
}

FLIGHTS = [
    {"destination": "Lisbon",    "code": "TP-833", "price": 180, "type": "direct",  "duration": "3h 05m"},
    {"destination": "Lisbon",    "code": "IB-310", "price": 150, "type": "1 stop",  "duration": "5h 10m"},
    {"destination": "Barcelona", "code": "VY-611", "price": 140, "type": "direct",  "duration": "1h 50m"},
    {"destination": "Barcelona", "code": "IB-220", "price": 125, "type": "1 stop",  "duration": "4h 00m"},
    {"destination": "Prague",    "code": "FR-721", "price": 110, "type": "direct",  "duration": "1h 55m"},
    {"destination": "Prague",    "code": "OS-410", "price": 145, "type": "1 stop",  "duration": "3h 45m"},
]

HOTELS = [
    {"destination": "Lisbon",    "name": "Baixa Stay",         "price_per_night": 145, "style": "central boutique hotel"},
    {"destination": "Lisbon",    "name": "River Rooms",         "price_per_night": 120, "style": "simple hotel near transit"},
    {"destination": "Barcelona", "name": "Born Hotel",          "price_per_night": 160, "style": "central design hotel"},
    {"destination": "Barcelona", "name": "Gracia Inn",          "price_per_night": 130, "style": "quiet hotel in a walkable district"},
    {"destination": "Prague",    "name": "Old Town House",      "price_per_night": 115, "style": "historic hotel near main sights"},
    {"destination": "Prague",    "name": "City Garden Hotel",   "price_per_night":  95, "style": "budget-friendly hotel with tram access"},
]

ACTIVITIES = [
    {"destination": "Lisbon",    "name": "Alfama food walk",                    "tag": "food",    "price": 35},
    {"destination": "Lisbon",    "name": "Belém and river tram day",            "tag": "culture", "price": 25},
    {"destination": "Barcelona", "name": "Gothic Quarter tapas evening",        "tag": "food",    "price": 40},
    {"destination": "Barcelona", "name": "Sagrada Família and modernism route", "tag": "culture", "price": 32},
    {"destination": "Prague",    "name": "Old Town walking tour",               "tag": "culture", "price": 18},
    {"destination": "Prague",    "name": "Czech dinner and jazz night",         "tag": "food",    "price": 30},
]

# ── Catalog helpers ───────────────────────────────────────────────────────────
# Simple filter functions so agents can look up options by destination.

def flights_for(destination: str) -> list[dict]:
    return [f for f in FLIGHTS if f["destination"] == destination]

def hotels_for(destination: str) -> list[dict]:
    return [h for h in HOTELS if h["destination"] == destination]

def activities_for(destination: str) -> list[dict]:
    return [a for a in ACTIVITIES if a["destination"] == destination]

def catalog_text() -> str:
    """Return the full catalog as a plain-text string suitable for a prompt."""
    lines = []
    for dest, info in DESTINATIONS.items():
        lines.append(f"Destination: {dest}")
        lines.append(f"  Best period : {info['best_period']}")
        lines.append(f"  Style       : {info['style']}")
        lines.append(f"  Notes       : {info['notes']}")
        lines.append("  Flights:")
        for f in flights_for(dest):
            lines.append(f"    - {f['code']} | {f['type']} | EUR {f['price']} | {f['duration']}")
        lines.append("  Hotels:")
        for h in hotels_for(dest):
            lines.append(f"    - {h['name']} | EUR {h['price_per_night']}/night | {h['style']}")
        lines.append("  Activities:")
        for a in activities_for(dest):
            lines.append(f"    - {a['name']} | {a['tag']} | EUR {a['price']}")
        lines.append("")
    return "\n".join(lines).strip()

CATALOG_TEXT = catalog_text()

print("USER_REQUEST:")
print(USER_REQUEST)
print("\nCatalog loaded — destinations:", list(DESTINATIONS.keys()))


USER_REQUEST:
Plan a 4-day spring trip from Rome.
Requirements:
- mid-range budget
- easy flights
- central hotel
- mix of food and culture
- simple daily plan

Catalog loaded — destinations: ['Lisbon', 'Barcelona', 'Prague']


## 2 · Define Team Roles and the Shared Project Board

### The shared project board

The board is a plain Python `dict` where keys are section names and values
start as empty strings.
Each teammate's job is to fill exactly one key.

```python
project_board = {
    "destination_and_period": "",   # filled by destination_designer
    "flight":                 "",   # filled by flight_specialist
    "hotel":                 "",   # filled by hotel_specialist
    "activities":            "",   # filled by activity_specialist
    "final_writeup":         "",   # filled by itinerary_writer
}
```

### Why `TeamSection` uses structured output

Each teammate returns a `TeamSection` with a single `content` field.
This makes it trivial to write the result back to the correct board key
without parsing free-text.

### Contrast with flat and hierarchical patterns

- In the **flat** pattern, any peer could modify any part of the plan.
- In the **hierarchical** pattern, the manager decides what each worker does.
- Here, each teammate has **standing ownership** of their section —
  the mapping is fixed in code, not decided at runtime.


In [2]:
# ── TeamSection schema ─────────────────────────────────────────────────────────
# One content field is all we need.
# The teammate's role is fully defined by which board key they write to,
# so we don't need the schema to carry role metadata.

class TeamSection(BaseModel):
    content: str = Field(description="The teammate's completed section of the project board.")

section_llm = model.with_structured_output(TeamSection)

# ── Team role registry ─────────────────────────────────────────────────────────
# Each entry maps a role name to:
#   field  → the board key this role is responsible for
#   prompt → the instruction given to that teammate
#
# The relay order is encoded in the list used in the next cell.
# Changing the order here would change which board entries are available
# to each downstream teammate — a design decision, not an accident.

TEAM_ROLES = {
    "destination_designer": {
        "field":  "destination_and_period",
        "prompt": (
            "Choose exactly one destination and one travel period from the catalog. "
            "Write only this section — do not mention flights, hotels, or activities."
        ),
    },
    "flight_specialist": {
        "field":  "flight",
        "prompt": (
            "Choose exactly one flight from the catalog that fits the chosen destination "
            "and the traveler's preferences. Write only the flight section."
        ),
    },
    "hotel_specialist": {
        "field":  "hotel",
        "prompt": (
            "Choose exactly one hotel from the catalog that fits the chosen destination "
            "and the traveler's preferences. Write only the hotel section."
        ),
    },
    "activity_specialist": {
        "field":  "activities",
        "prompt": (
            "Choose two or three activities from the catalog that match the traveler's "
            "interests (food and culture mix). Write only the activities section."
        ),
    },
    "itinerary_writer": {
        "field":  "final_writeup",
        "prompt": (
            "The destination, flight, hotel, and activities sections are complete. "
            "Turn the full project board into one clean, client-ready itinerary. "
            "Write only the final write-up section."
        ),
    },
}

# ── Teammate runner ────────────────────────────────────────────────────────────
def run_teammate(role_name: str, board: dict) -> TeamSection:
    """
    Call one teammate.

    Key team-pattern properties visible here:
    - The teammate receives the full board — earlier sections already filled.
    - The teammate writes ONLY their own section (enforced by the prompt).
    - The teammate does not know about other teammates or their roles.
    """
    role = TEAM_ROLES[role_name]
    # Format the current board state as readable text for the prompt.
    board_text = "\n".join(
        f"  {k}: {v if v else '(not filled yet)'}"
        for k, v in board.items()
    )

    messages = [
        SystemMessage(content=(
            "You are a specialist teammate in a modular travel-agency MAS.\n"
            "You own exactly one section of the shared project board. "
            "Read the board, then write your section only.\n"
            "Do not invent options outside the catalog.\n\n"
            f"Your role: {role['prompt']}"
        )),
        HumanMessage(content=(
            f"Traveler request:\n{USER_REQUEST}\n\n"
            f"Catalog:\n{CATALOG_TEXT}\n\n"
            f"Shared project board (current state):\n{board_text}\n\n"
            f"Write the '{role['field']}' section now."
        )),
    ]
    return section_llm.invoke(messages)

print("Team roles defined:", list(TEAM_ROLES.keys()))


Team roles defined: ['destination_designer', 'flight_specialist', 'hotel_specialist', 'activity_specialist', 'itinerary_writer']


## 3 · Execute the Team Relay

The workflow is a fixed pipeline.
The relay order is hard-coded because the task has a natural dependency structure:

1. You must know the **destination** before you can pick a flight.
2. You must know the **destination** before you can pick a hotel.
3. You must know the **destination** before you can pick activities.
4. You must know **everything** before writing the final itinerary.

> This explicit dependency order — enforced in code rather than by a manager —
> is what makes the team pattern feel reliable but inflexible.
> Adding a new step is as simple as adding a new entry to `TEAM_ROLES`
> and inserting the role name in the relay list.


In [3]:
# ── Shared project board ───────────────────────────────────────────────────────
# Starts empty. Each teammate fills exactly one key.
project_board = {
    "destination_and_period": "",
    "flight":                 "",
    "hotel":                  "",
    "activities":             "",
    "final_writeup":          "",
}

# ── Relay execution ────────────────────────────────────────────────────────────
# The relay order encodes the task's dependency structure.
# Earlier teammates cannot be skipped — downstream teammates depend on their output.
RELAY_ORDER = [
    "destination_designer",
    "flight_specialist",
    "hotel_specialist",
    "activity_specialist",
    "itinerary_writer",
]

for role_name in RELAY_ORDER:
    field_name = TEAM_ROLES[role_name]["field"]

    # Run the teammate — they read the full board and write one section.
    section = run_teammate(role_name, project_board)

    # Write the result back to the board.
    # Only the field this role owns changes; all other fields stay intact.
    project_board[field_name] = section.content

    print(f"[{role_name}] → wrote '{field_name}' ({len(section.content)} chars)")

# ── Results ────────────────────────────────────────────────────────────────────
print("\n" + "="*60)
print("SHARED PROJECT BOARD (final state)")
print("="*60)
for field, value in project_board.items():
    print(f"\n── {field} ──")
    print(value)

print("\n" + "="*60)
print("FINAL ITINERARY (itinerary_writer's section)")
print("="*60)
print(project_board["final_writeup"])


[destination_designer] → wrote 'destination_and_period' (50 chars)
[flight_specialist] → wrote 'flight' (187 chars)
[hotel_specialist] → wrote 'hotel' (333 chars)
[activity_specialist] → wrote 'activities' (314 chars)
[itinerary_writer] → wrote 'final_writeup' (790 chars)

SHARED PROJECT BOARD (final state)

── destination_and_period ──
Destination: Lisbon
Best travel period: April–June

── flight ──
Flight: TP-833 | direct | EUR 180 | 3h 05m

This direct flight from Rome to Lisbon offers an easy and convenient travel option, fitting the mid-range budget and preference for simplicity.

── hotel ──
Hotel: Baixa Stay | EUR 145/night | central boutique hotel

Baixa Stay is a centrally located boutique hotel in Lisbon, offering a comfortable mid-range option that fits the traveler's preference for a central hotel. Its location in the heart of the city makes it ideal for exploring Lisbon's food and cultural attractions with ease.

── activities ──
Activities:
- Alfama food walk | food | EUR

## 4 · Key Takeaways

| What you saw | Why it matters |
|---|---|
| Each role owns **exactly one board section** | Responsibilities never overlap — easy to debug |
| Teammates run in a **fixed relay order** | Dependencies are explicit, not managed at runtime |
| Each teammate reads the **full board** | Later specialists benefit from all earlier decisions |
| No manager, no voting | The workflow is the orchestration |

### How this differs from the other notebooks
- **vs Flat (nb 1):** roles are strict ownership boundaries, not soft tendencies
- **vs Hierarchical (nb 2):** there is no central planner — the pipeline is fixed in code
- **vs Society (nb 3):** teammates build one plan collaboratively rather than choosing between fixed options

### When the team pattern struggles
- **Cascading errors** — if the destination designer picks poorly, every downstream
  teammate inherits the mistake with no mechanism to correct it
- **Rigid pipeline** — adding conditional logic (e.g., "skip hotel if camping")
  requires code changes, not just prompt changes
- **No revisiting** — once a section is written it is not revised,
  even if a later teammate notices a mismatch

---

## Series Summary

| | Flat | Hierarchical | Society | Team |
|---|---|---|---|---|
| Authority | None | Manager | Social rules | Section ownership |
| Communication | Shared board | Star (worker→manager) | Public ballot | Sequential relay |
| Stops when | All peers agree | Manager decides | Majority vote | Pipeline complete |
| Best for | Open-ended brainstorm | Decomposable tasks | Multi-criteria selection | Clean parallel specialisation |
| Main risk | Contradiction loops | Manager bottleneck | Vote deadlock | Error propagation |
